# Lab 4: Parameter-efficient fine-tuning

Fine-tuning all parameters of pre-trained language models can be resource-intensive. Because of this, current research in natural language processing is looking into developing methods for adapting models to downstream tasks without full fine-tuning. These methods only tune a small number of model parameters while yielding performance comparable to that of a fully fine-tuned model.

In this lab, you will implement LoRA, one of the most well-known methods for parameter-efficient fine-tuning. LoRA stands for “Low-Rank Adaptation of Large Language Models” and was originally described in a research article by [Hu et al. (2021)](https://arxiv.org/abs/2106.09685).

Along the way, you will earn experience with [Hugging Face Transformers](https://huggingface.co/docs/transformers/en/index), a state-of-the-art library for training and deploying language models, as well as with several related libraries. In particular, you will learn a best-practice workflow for downloading a Transformer model and fine-tuning it on the downstream task of binary sentiment classification.

*Tasks you can choose for the oral exam are marked with the graduation cap 🎓 emoji.*

## Dataset

The data for this lab comes from the [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/). The full dataset consists of 50,000 highly polar movie reviews collected from the Internet Movie Database (IMDB). Here, we use a random sample consisting of 2,000 reviews for training and 500 reviews for evaluation.

To load the dataset, we use the [Hugging Face Datasets](https://huggingface.co/docs/datasets/en/index) library.

In [1]:
from datasets import load_dataset

imdb_dataset = load_dataset(
    "csv", data_files={"train": "train.csv", "eval": "eval.csv"}
)

imdb_dataset

/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['index', 'review', 'label'],
        num_rows: 2000
    })
    eval: Dataset({
        features: ['index', 'review', 'label'],
        num_rows: 500
    })
})

As we can see, each sample in the dataset is a record with three fields: an internal index (`index`, an integer), the text of the review (`review`, a string), and the sentiment label (`label`, an integer – 1&nbsp;for “positive” and 0&nbsp;for “negative” sentiment).

Here is an example record:

In [2]:
imdb_dataset["train"][645]

{'index': 2981,
 'review': 'Brilliant execution in displaying once and for all, this time in the venue of politics, of how "good intentions do actually pave the road to hell". Excellent!',
 'label': 1}

## Tokeniser

As our pre-trained language model, we will use [DistilBERT](https://huggingface.co/docs/transformers/en/model_doc/distilbert), a compact encoder model with 40% less parameters than BERT base. DistilBERT is not actually a *large* language model by modern standards and thus does not benefit as much from parameter-efficient fine-tuning as other models. However, it has the benefit of being light and fast, and can be run even on consumer hardware.

To feed the movie reviews to DistilBERT, we need to tokenise them and encode the resulting tokens as integers in the model vocabulary. We start by loading the DistilBERT tokeniser using the [Auto classes](https://huggingface.co/docs/transformers/en/model_doc/auto):

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

We then create a tokenised version of the dataset:

In [4]:
def tokenize_function(batch):
    return tokenizer(batch["review"], padding=True, truncation=True)


tokenized_imdb_dataset = imdb_dataset.map(tokenize_function, batched=True)

tokenized_imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['index', 'review', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    eval: Dataset({
        features: ['index', 'review', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 500
    })
})

As we can see, tokenising adds two additional fields to each review: `input_ids` is the list of token ids corresponding to the review, and `attention_mask` is the list of indices specifying which tokens the encoder should attend to.

To avoid trouble when fine-tuning the model later, the next cell disables tokeniser parallelism.

In [5]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Trainer

In this section, we will set up our workflow for training and evaluating DistilBERT models. The central component in this workflow is the [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer), which provides extensive configuration options. Here, we leave most of these options at their default value. Two changes we *do* make are to enable evaluation of the trained model after each epoch, and to log the training and evaluation loss after every 5&nbsp;training steps (the default is 500).

In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="tmp_trainer",
    eval_strategy="epoch",
    logging_steps=5,
)

In addition to the loss, we also track classification accuracy. For this we import the [Hugging Face Evaluate](https://huggingface.co/docs/evaluate/en/index) library and define a small helper function `compute_metrics()` that the trainer will call after each epoch.

In [7]:
import evaluate

accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In the next cell we define a convenience function `make_trainer()` that creates a readily-configured trainer for a specified model (*model*). We will use this trainer both to train the model on the training section of the tokenised review dataset, and to evaluate it on the evaluation section.

In [8]:
from transformers import Trainer


def make_trainer(model):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_imdb_dataset["train"],
        eval_dataset=tokenized_imdb_dataset["eval"],
        compute_metrics=compute_metrics,
    )
    return trainer

## Full fine-tuning

In the rest of this notebook, we will work our way to the implementation of LoRA, and compare LoRA to traditional fine-tuning methods. Our first point of reference is a fully fine-tuned DistilBERT model.

We start by loading the pre-trained model:

In [9]:
from transformers import AutoModelForSequenceClassification

pretrained_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

pretrained_model

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1775.63it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


The architecture of DistilBERT is that of a standard Transformer encoder with an embedding layer (`embeddings`) followed by a stack of six Transformer blocks (`transformer`) and a feedforward network with two linear layers (`pre_classifier` and `classifier`) and a final dropout layer (`dropout`).

### 🧩 Task 4.01: Counting the number of trainable parameters

One relevant measure in the context of parameter-efficient fine-tuning is the number of parameters that need to be changed when training a model. Your first task in this lab is to write a function `num_trainable_parameters()` that calculates this number for a given model.

In [13]:
def num_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(num_trainable_parameters(pretrained_model))

66955010


The function should implement the following specification:

> **num_trainable_parameters** (*model*)
>
> Returns the number of float-valued trainable parameters in the specified *model* as an integer.

#### 👍 Hint

The term *parameter* can refer to either complete tensors or the individual elements of these tensors. For example, a linear layer created by `nn.Linear(3, 5)` has 2&nbsp;tensor-valued parameters (a weight matrix and a bias vector) and 20&nbsp;float-valued parameters (the elements of these tensors). To get the tensor-valued parameters of a model, you can use the [`parameters()`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.parameters) method. A parameter is *trainable* if it requires gradient.

#### 🤞 Test your code

To test your code, apply your function to the pre-trained model. The correct number of float-valued trainable parameters for this model is 66,955,010.

### Fine-tuning

When we load the pre-trained model, the Hugging Face Transformers library warns us that the weights of the feedforward network have not yet been trained. To do so, in the next cell, we pass the pre-trained model to a trainer and initiate the fine-tuning process.

**⚠️ Please note that fine-tuning the model will take some time! ⚠️**

You can work on the other problems in this lab while you are waiting.

In [14]:
finetuned_trainer = make_trainer(pretrained_model)

finetuned_trainer.train()

/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.589222,0.453527,0.866000
2,0.004859,0.425252,0.914000
3,0.002654,0.469918,0.916000


/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.76it/s]
/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.43it/s]
/Users/dennisjohansson/Skola/P

TrainOutput(global_step=750, training_loss=0.17483655247402688, metrics={'train_runtime': 1570.4291, 'train_samples_per_second': 3.821, 'train_steps_per_second': 0.478, 'total_flos': 794804391936000.0, 'train_loss': 0.17483655247402688, 'epoch': 3.0})

Because full fine-tuning is so resource-intensive, we save the fine-tuned model to disk:

In [15]:
finetuned_trainer.save_model("finetuned")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.33it/s]


Later in this notebook, whenever you need the fully fine-tuned version of the model, you can load it as follows:

In [16]:
finetuned_model = AutoModelForSequenceClassification.from_pretrained("finetuned")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1410.93it/s, Materializing param=pre_classifier.weight]                                  


### Convenience functions

Because we will repeat the steps we just took to fine-tune the pre-trained model several times in this notebook, we define two convenience functions:

In [17]:
def train(model):
    print("Number of trainable parameters:", num_trainable_parameters(model))
    trainer = make_trainer(model)
    trainer.train()
    return model

In [18]:
def evaluate(model):
    trainer = make_trainer(model)
    return trainer.evaluate()

## Tuning the final layers only

If full fine-tuning marks one end of the complexity spectrum, the other end is marked by only tuning the final layers of the transformer – the *head* of the model. In the case of DistilBERT, the head consists of the `pre_classifier` and `classifier` layers.

### 🧩 Task 4.02: Head-tuning

Implement the head-tuning strategy by coding the following function:

In [21]:
def make_headtuned_model():
    model = finetuned_model

    # freeze all
    for p in model.parameters():
        p.requires_grad = False

    # unfreeze head params by name
    head_keywords = ("classifier", "score", "classification_head")
    for name, p in model.named_parameters():
        if any(k in name for k in head_keywords):
            p.requires_grad = True

    return model

Here is the specification of this function:

> **make_headtuned_model** ()
>
> Returns a model that is identical to the pre-trained model, except that the head layers have been trained on the sentiment data. (The other parameters of the pre-trained model are left untouched.)

#### 👍 Hint

You freeze a parameter by setting its `requires_grad`-attribute to `False`.

Once you have an implementation of the head-tuning strategy, evaluate it on the evaluation data. How much accuracy do we lose when only training the final layers of the pre-trained model, compared to full fine-tuning?

In [22]:
headtuned_model = make_headtuned_model()
print(num_trainable_parameters(headtuned_model))

592130


#### 🤞 Test your code

If you configured your model correctly, `num_trainable_parameters()` should show 592,130 trainable parameters.

For future reference, we also save the head-tuned model:

In [23]:
make_trainer(headtuned_model).save_model("headtuned")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.51s/it]


## Layer surgery

LoRA works by “wrapping” frozen layers from the pre-trained Transformer model inside adapter modules. Conventionally, this wrapping is only applied to the linear layers that transform the queries and values in the self-attention mechanism. To implement the wrapping, we need functions to extract and replace layers in a model. Your task in this section is to code these functions.

### 🎓 Task 4.03: Extracting layers

Code a function that extracts the query and value linear layers from a DistilBERT model:

In [27]:
def extract(model):
    layers = {}

    # DistilBERT has 6 transformer blocks by default (0 to 5)
    for i in range(6):
        q_name = f"distilbert.transformer.layer.{i}.attention.q_lin"
        v_name = f"distilbert.transformer.layer.{i}.attention.v_lin"

        layers[q_name] = model.get_submodule(q_name)
        layers[v_name] = model.get_submodule(v_name)

    return layers

extracted_layers = extract(finetuned_model)
print(sum(p.numel() for layer in extracted_layers.values() for p in layer.parameters()))

7087104


Implement this function to match the following specification:

> **extract** (*model*)
>
> Takes a DistilBERT model (*model*) and extracts the query and value linear layers from each block of the Transformer. Returns a dictionary mapping the DistilBERT module names of these layers to the layers themselves (instances of `nn.Linear`).

#### 👍 Hint

As we saw earlier, the DistilBERT model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. Use [`get_submodule()`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.get_submodule) to retrieve a layer by name. You can hard-wire the names of the layers you want to extract.

#### 🤞 Test your code

To test your code, check the number of trainable float-valued parameters in the extracted layers. This number should be 7,087,104.

### 🎓 Task 4.04: Replacing layers

Next, code the inverse of the `extract()` function to replace selected layers of a module using a dictionary of named layers.

In [28]:
def replace(model, named_layers):
    for full_name, layer in named_layers.items():
        parts = full_name.split(".")
        parent = model

        for p in parts[:-1]:
            parent = getattr(parent, p)

        setattr(parent, parts[-1], layer)

    return model

Implement this function to match the following specification:

> **replace** (*model*, *named_layers*)
>
> Takes a DistilBERT model (*model*) and a dictionary in the format returned by `extract()` (*named_layers*) and injects the extracted layers into the model. More specifically, suppose that *named_layers* contains a key–value pair `(name, layer)`. Then the function replaces the submodule of *model* addressed by the fully-qualified string name `name` by the layer `layer`. Returns the modified model.

#### 👍 Hint

Use [`getattr()`](https://docs.python.org/3/library/functions.html#getattr) and [`setattr()`](https://docs.python.org/3/library/functions.html#setattr) to return or set the value of a named submodule.

#### 🤞 Test your code

To test your implementation, write code that (1)&nbsp;extracts the query and value linear layers from the fine-tuned model; (2)&nbsp;replaces these layers with clones with random weights; and (3)&nbsp;replaces these layers again with the original versions. Evaluating the modified model after step&nbsp;(2) should yield a near-random accuracy. Evaluating it again after step&nbsp;(3) should yield the original accuracy.

The following function should be helpful. It clones a linear layer, copying the weights and the bias from the original.

In [29]:
import torch.nn as nn
import copy
import torch


def clone_linear(original):
    out_features, in_features = original.weight.shape
    copy = nn.Linear(in_features, out_features)
    copy.load_state_dict(original.state_dict())
    return copy


model = copy.deepcopy(finetuned_model)
orig_layers = extract(model)

# Skapa "random weights"-lager genom att klona och sen randomisera parametrarna
random_layers = {}
for name, layer in orig_layers.items():
    c = clone_linear(layer) 
    with torch.no_grad():
        c.weight.normal_()  
        if c.bias is not None:
            c.bias.normal_()
    random_layers[name] = c

model_random = replace(model, random_layers)

print("Accuracy after random replacement:")
print(evaluate(model_random))  

model_restored = replace(model_random, orig_layers)

print("Accuracy after restoring originals:")
print(evaluate(model_restored))

Accuracy after random replacement:


/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.7304400205612183, 'eval_model_preparation_time': 0.0038, 'eval_accuracy': 0.472, 'eval_runtime': 53.7687, 'eval_samples_per_second': 9.299, 'eval_steps_per_second': 1.172}
Accuracy after restoring originals:


/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.46991780400276184, 'eval_model_preparation_time': 0.0008, 'eval_accuracy': 0.916, 'eval_runtime': 51.0392, 'eval_samples_per_second': 9.796, 'eval_steps_per_second': 1.234}


## Low-rank approximation

The basic idea behind LoRA is to conceptualise fine-tuned weights as a sum $W_0 + \Delta W$ of the weights from the pre-trained model, $W_0$, and a low-rank update matrix $\Delta W$. The goal of fine-tuning, then, is to learn the update matrix; this happens in the adapter layers.

Before we get to the implementation of the LoRA adapter layers, we first check to what extent the assumption that fine-tuning can be described by low-rank matrices holds true for DistilBERT. To do so, we will “cheat” and replace the query and value linear layers of the head-tuned model with low-rank approximations. The technical key to this is the truncated singular value decomposition (SVD).

### 🎓 Task 4.05: Low-rank matrix approximation

Your first task in this section is to implement the low-rank matrix approximation.

In [31]:
def approximate(matrix, rank):
    U, S, Vh = torch.linalg.svd(matrix, full_matrices=False)

    U_r = U[:, :rank]
    S_r = S[:rank]
    Vh_r = Vh[:rank, :]

    # Reconstruct low-rank approximation
    return U_r @ torch.diag(S_r) @ Vh_r

Implement this function to match the following specification:

> **approximate** (*matrix*, *rank*)
>
> Takes a 2D-tensor (*matrix*) and an integer rank $r$ (*rank*), computes the truncated SVD with rank $r$ on the tensor, and returns the corresponding low-rank approximation matrix.

#### 👍 Hint

If you need a refresher on the low-rank matrix approximation, read the corresponding section from the Wikipedia article on the [Singular value decomposition](https://en.wikipedia.org/wiki/Singular_value_decomposition#Low-rank_matrix_approximation). The truncated SVD is an extension of the full SVD; the latter can be computed using [`torch.linalg.svd()`](https://pytorch.org/docs/stable/generated/torch.linalg.svd.html).

#### 🤞 Test your code

To test your code, run the following cell. It creates a matrix `original` with rank $r \leq 8$ and after that the rank-$8$ approximation matrix `approximation`. You should find that the distance between the two matrices is very low.

In [32]:
original = torch.rand(768, 8) @ torch.rand(8, 384)
approximation = approximate(original, 8)
d = torch.dist(original, approximation)
print("dist: ", d.item())

dist:  0.000558929517865181


### 🎓 Task 4.06: Approximated fine-tuned model (version 1)

In the next step, your task is to construct a version of the head-tuned model in which every query and value linear layer is replaced by a low-rank approximation of the corresponding layer from the fully fine-tuned model.

In [33]:
def make_approximated_model_1(rank):
    model = make_headtuned_model()
    ft_layers = extract(finetuned_model) 

    # Repplace layers with low-rank weights
    approx_layers = {}
    for name, layer in ft_layers.items():
        out_features, in_features = layer.weight.shape

        new_layer = nn.Linear(in_features, out_features, bias=(layer.bias is not None))
        with torch.no_grad():
            W = layer.weight.data
            W_r = approximate(W, rank)
            new_layer.weight.copy_(W_r)

            if layer.bias is not None:
                new_layer.bias.copy_(layer.bias.data)

        approx_layers[name] = new_layer

    model = replace(model, approx_layers)
    return model

Here is the specification of this function:

> **make_approximated_model_1** (*rank*)
>
> Takes an integer rank $r$ (*rank*) and returns a version of the head-tuned model in which every query and value linear layer is replaced by its $r$-approximated corresponding layer from the fully fine-tuned model.

Run the next cell to evaluate your model for different rank values. Start with the full rank and then halve the rank in each step. What is the lowest rank that still gives you a higher accuracy than the head-tuned model?

In [37]:
approximated_model_1 = make_approximated_model_1(768)
print(evaluate(approximated_model_1))

approximated_model_2 = make_approximated_model_1(384)
print(evaluate(approximated_model_2))

approximated_model_3 = make_approximated_model_1(192)
print(evaluate(approximated_model_3))

approximated_model_4 = make_approximated_model_1(96)
print(evaluate(approximated_model_4))

approximated_model_5 = make_approximated_model_1(48)
print(evaluate(approximated_model_5))

approximated_model_6 = make_approximated_model_1(24)
print(evaluate(approximated_model_6))

approximated_model_7 = make_approximated_model_1(12)
print(evaluate(approximated_model_7))

approximated_model_8 = make_approximated_model_1(6)
print(evaluate(approximated_model_8))

approximated_model_9 = make_approximated_model_1(3)
print(evaluate(approximated_model_9))

/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.6869566440582275, 'eval_model_preparation_time': 0.0022, 'eval_accuracy': 0.474, 'eval_runtime': 142.1358, 'eval_samples_per_second': 3.518, 'eval_steps_per_second': 0.443}


/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.6869566440582275, 'eval_model_preparation_time': 0.0009, 'eval_accuracy': 0.474, 'eval_runtime': 100.7704, 'eval_samples_per_second': 4.962, 'eval_steps_per_second': 0.625}


/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


...

{'eval_loss': 0.6869566440582275, 'eval_model_preparation_time': 0.001, 'eval_accuracy': 0.474, 'eval_runtime': 160.9636, 'eval_samples_per_second': 3.106, 'eval_steps_per_second': 0.391}


/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.686956524848938, 'eval_model_preparation_time': 0.0019, 'eval_accuracy': 0.474, 'eval_runtime': 275.9738, 'eval_samples_per_second': 1.812, 'eval_steps_per_second': 0.228}


/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.686956524848938, 'eval_model_preparation_time': 0.0014, 'eval_accuracy': 0.474, 'eval_runtime': 152.9101, 'eval_samples_per_second': 3.27, 'eval_steps_per_second': 0.412}


### 🎓 Task 4.07: Approximated fine-tuned model (version 2)

In the approximated model from the previous section, the truncated SVD is applied to the full weight matrix of the fine-tuned model: $W_0 + \Delta W$. In LoRA, the low-rank approximation only applies to the *update matrix* $\Delta W$, i.e., the difference between the fully fine-tuned weights and the pre-trained weights.

In [38]:
def make_approximated_model_2(rank):
    model = make_headtuned_model()
    base_layers = extract(pretrained_model)   
    ft_layers   = extract(finetuned_model)    

    # Build low-rank-update replacements
    new_layers = {}
    for name in ft_layers.keys():
        base = base_layers[name]
        ft   = ft_layers[name]

        out_features, in_features = ft.weight.shape
        new_layer = nn.Linear(in_features, out_features, bias=(ft.bias is not None))

        with torch.no_grad():
            W0  = base.weight.data
            Wft = ft.weight.data
            dW  = Wft - W0
            dW_r = approximate(dW, rank)
            Wnew = W0 + dW_r
            new_layer.weight.copy_(Wnew)

            if ft.bias is not None:
                new_layer.bias.copy_(ft.bias.data)

        new_layers[name] = new_layer

    model = replace(model, new_layers)
    return model

Implement the function to match the following specification:

> **make_approximated_model_2** (*rank*)
>
> Takes an integer rank $r$ (*rank*) and returns a version of the head-tuned model in which the weight matrix of every query and value linear layer is replaced by the sum $W_0 + \Delta W$, where $W_0$ is the weight matrix of the pre-trained model and $\Delta W$ is the rank-$r$ approximation of the update matrix, i.e., the difference between the fully fine-tuned weights and the pre-trained weights.

Run the next cell to evaluate your model for different rank values. Start with the rank from the approximated model from the previous section and then halve the rank in each step. What is the lowest rank that still gives you a higher accuracy than the head-tuned model?

In [39]:
approximated_model_2 = make_approximated_model_2(768)

evaluate(approximated_model_2)

/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.6869566440582275,
 'eval_model_preparation_time': 0.0016,
 'eval_accuracy': 0.474,
 'eval_runtime': 101.1849,
 'eval_samples_per_second': 4.941,
 'eval_steps_per_second': 0.623}

In [40]:
approximated_model_2 = make_approximated_model_2(3)

evaluate(approximated_model_2)

/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.5309256315231323,
 'eval_model_preparation_time': 0.0023,
 'eval_accuracy': 0.884,
 'eval_runtime': 186.9526,
 'eval_samples_per_second': 2.674,
 'eval_steps_per_second': 0.337}

Rank 3 still gives higher accuracy than head-tuned

## Low-Rank Adaptation (LoRA)

In this section, you will implement the LoRA adapters and fine-tune the adapted model.

### 🎓 Task 4.08: Implement the adapter

A LoRA adapter implements the forward function

$$
y = x W_0 + x \Delta W = x W_0 + x A B
$$

where $W_0$ is a linear transformation from the pre-trained model and $\Delta W$ is a learned update matrix, deconstructed into the product $AB$ of two rank-$r$ matrices $A$ and $B$. LoRA scales the update matrix $\Delta W$ by a factor of $\alpha / r$, where $\alpha$ is a hyperparameter. (To keep the formula tidy, we ignore the fact that the linear transformation in the pre-trained model may additionally include a bias.)

In [41]:
import torch.nn as nn

class LoRA(nn.Module):
    def __init__(self, pretrained, rank=12, alpha=24):
        super().__init__()
        self.pretrained = pretrained
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        # Freeze the pretrained layer
        for p in self.pretrained.parameters():
            p.requires_grad = False

        in_features = pretrained.in_features
        out_features = pretrained.out_features

        # A ~ N(0,1), B = 0
        self.A = nn.Parameter(torch.randn(in_features, rank))
        self.B = nn.Parameter(torch.zeros(rank, out_features))

    def forward(self, x):
        base = self.pretrained(x)                        
        update = (x @ self.A @ self.B) * self.scaling   
        return base + update

Your code must comply with the following specification:

**__init__** (*self*, *pretrained*, *rank* = 12, *alpha* = 24)

> Initialises the LoRA adapter. This sets up the matrices $A$ and $B$ from the equation above. The matrix $A$ is initialised with random weights from a standard normal distribution; the matrix $B$ is initialised with zeros. The argument *pretrained* is the linear layer from the pre-trained model that should be adapted. The arguments *rank* and *alpha* are the rank $r$ and the hyperparameter $\alpha$ in the equation above.

**forward** (*self*, *x*)

> Sends an input *x* through the adapter, implementing the equation above.

### 🎓 Task 4.09: Inject the adapter into the pre-trained model

The final step is to construct an adapted model by injecting the LoRA adapters into the pre-trained model.

In [42]:
def make_lora_model(rank):
    alpha = 2 * rank

    model = finetuned_model

    # Freeze everything
    for p in model.parameters():
        p.requires_grad = False

    # Unfreeze the classification head
    for attr in ["classifier", "score", "classification_head"]:
        if hasattr(model, attr):
            for p in getattr(model, attr).parameters():
                p.requires_grad = True
            break

    # Wrap q_lin and v_lin with LoRA adapters
    qv = extract(model)
    wrapped = {}
    for name, lin in qv.items():
        wrapped[name] = LoRA(lin, rank=rank, alpha=alpha) 

    model = replace(model, wrapped)

    return model

Implement the function to match the following specification:

> **make_lora_model** (*rank*)
>
> Returns a model that is identical to the pre-trained model, except that the query and value linear layers have been wrapped in LoRA adapters, and the LoRA adapters and the head layers of the pre-trained model have been trained on the sentiment data. (The other parameters of the pre-trained model are left untouched.) The rank of the adapters is specified by the argument *rank*. The *alpha* value of the adapters is set to twice the rank (a common rule of thumb).

Run the next cell to evaluate your model for $r = 6$ and $\alpha = 12$. How many trainable parameters does the adapted model have? What accuracy do you get? How do these value relate to the number of trainable parameters and accuracy of the fully fine-tuned model, in terms of percentages?

In [43]:
lora_model = make_lora_model(6)
print(num_trainable_parameters(lora_model))
evaluate(lora_model)

112130


/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.5309256315231323,
 'eval_model_preparation_time': 0.003,
 'eval_accuracy': 0.884,
 'eval_runtime': 127.6823,
 'eval_samples_per_second': 3.916,
 'eval_steps_per_second': 0.493}

## Task 4.09 Answer

- **Trainable parameters (LoRA model):** 112,130  
- **Eval accuracy (LoRA model):** 0.884  

### Comparison to full fine-tuning
Full fine-tuning has **66,955,010** trainable parameters.

- **Trainable parameter ratio:** $(\frac{112{,}130}{66{,}955{,}010} \approx 0.001675)$  
  $(\Rightarrow) \approx 0.1675%$ of the trainable parameters

### Accuracy comparison (needs your full fine-tuned accuracy)
Let $(acc_{\text{full}})$ be the eval accuracy of the fully fine-tuned model.

- **Relative accuracy:** $(100 \cdot \frac{0.884}{acc_{\text{full}}}\%)$  
- **Accuracy drop (percentage points):** $(acc_{\text{full}} - 0.884)$

## Alternatives to Transformer-based models

Even with methods for parameter-efficient fine-tuning, applying DistilBERT and other Transformer-based models comes at a significant cost – an investment that does not always pay off. In the final task of this lab, we ask you to explore a more traditional approach to classification and contrast it with the pre-training/fine-tuning approach of neural language models.

### 🎓 Task 4.10: Comparing with a non-neural classifier

Browse the web to find a tutorial on how to apply a classifier from the [scikit-learn](https://scikit-learn.org/stable/) library to the problem of sentiment classification and implement the method here in this notebook. We suggest you use [multinomial Naive Bayes](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html) or [logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html). (Once you have code for one method, it is easy to switch to the other.) Evaluate your chosen classifier on the IMDB dataset.

Questions to consider:

* Which classifier did you try? What results did you get? How long did it take you to train and run the classifier?
* What is your perspective on the trade-off between accuracy and resource requirements between the two approaches?
* What did you learn? How, exactly, did you learn it? Why does this learning matter?

In [48]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from time import perf_counter

# Load data
train_df = pd.read_csv("train.csv")
eval_df  = pd.read_csv("eval.csv")

X_train, y_train = train_df["review"].astype(str), train_df["label"]
X_eval,  y_eval  = eval_df["review"].astype(str),  eval_df["label"]

# Pipeline: TF-IDF -> Logistic Regression
clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=50_000,      
        ngram_range=(1, 2)        
    )),
    ("logreg", LogisticRegression(
        max_iter=1000,
        n_jobs=-1
    ))
])

t0 = perf_counter()
clf.fit(X_train, y_train)
train_time = perf_counter() - t0

t0 = perf_counter()
pred = clf.predict(X_eval)
eval_time = perf_counter() - t0

acc = accuracy_score(y_eval, pred)
print("Train time (s):", round(train_time, 3))
print("Eval time (s):", round(eval_time, 3))
print("Eval accuracy:", round(acc, 4))
print()
print(classification_report(y_eval, pred, digits=4))

Train time (s): 0.6
Eval time (s): 0.069
Eval accuracy: 0.832

              precision    recall  f1-score   support

           0     0.8809    0.7871    0.8313       263
           1     0.7887    0.8819    0.8327       237

    accuracy                         0.8320       500
   macro avg     0.8348    0.8345    0.8320       500
weighted avg     0.8372    0.8320    0.8320       500



/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


## Task 4.10: Answer

### Classifier and setup
I used a **scikit-learn Logistic Regression** classifier with **TF-IDF** text features. The model learns a weight for each word/phrase feature and predicts positive vs negative sentiment from a weighted sum of these features.

### Results (IMDB eval set)
- **Train time:** 0.606 s  
- **Eval time:** 0.050 s  
- **Eval accuracy:** 0.832  

(Class 0: precision 0.8809, recall 0.7871; Class 1: precision 0.7887, recall 0.8819.)

### Trade-off: accuracy vs resources
- The Logistic Regression + TF-IDF model is **extremely fast** to train and evaluate on CPU and has low resource requirements.
- The Transformer-based approach requires **much more compute and time**, but reached **higher accuracy**.
- So the classic model gives a strong baseline at very low cost, while Transformers give better accuracy when you can afford the extra resources.

### What we learned
I learned that a simple TF-IDF representation + a linear classifier can already perform well on sentiment classification because many sentiment cues are captured by informative words and short phrases. This matters because it provides a cheap, fast baseline and shows that you don’t always need heavy fine-tuning to get reasonable performance.

**🥳 Congratulations on finishing this lab! 🥳**